In [15]:
import pandas as pd
import numpy as np
import json
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [16]:
#Load process Dataset

file_path = "../data/processed/processed_interactions.csv"

df = pd.read_csv(file_path)

print("Processed dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Processed dataset loaded successfully.
Shape: (446844, 11)


,interaction_order,student_id,session_id,problem_id,concept_name,correct,attempt_count,hint_count,hint_total,response_time_ms,opportunity
0,21617623,14,263599,93383,Circle Graph,0,1,2,2,26271,1
1,21617623,14,263599,93383,Percent Of,0,1,2,2,26271,1
2,21617632,14,263599,93407,Circle Graph,1,1,0,2,29123,2
3,21617632,14,263599,93407,Percent Of,1,1,0,2,29123,2
4,21617641,14,263599,93400,Circle Graph,0,1,2,2,13779,3


In [17]:
#Basic Validation

required_columns = [
    "interaction_order",
    "student_id",
    "session_id",
    "problem_id",
    "concept_name",
    "correct",
    "attempt_count",
    "hint_count",
    "hint_total",
    "response_time_ms",
    "opportunity"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are available.")

All required columns are available.


In [18]:
# Part A : Generate Short-Term Memory Table

# Short-term memory should represent the latest session state for each student.

#Create Short-Term Memory for All Students

# Find the latest session for each student based on interaction order

latest_interactions = df.sort_values("interaction_order").groupby("student_id").tail(1)

latest_sessions = latest_interactions[["student_id", "session_id"]]

short_term_rows = []

for _, row in latest_sessions.iterrows():
    student_id = row["student_id"]
    session_id = row["session_id"]
    
    session_df = df[
        (df["student_id"] == student_id) &
        (df["session_id"] == session_id)
    ].copy()
    
    current_concept = session_df.sort_values("interaction_order")["concept_name"].iloc[-1]
    recent_accuracy = session_df["correct"].mean()
    average_attempts = session_df["attempt_count"].mean()
    recent_hint_usage = session_df["hint_count"].sum()
    interaction_count = len(session_df)
    
    if recent_accuracy >= 0.75:
        session_status = "good_progress"
    elif recent_accuracy >= 0.50:
        session_status = "moderate_progress"
    else:
        session_status = "needs_support"
    
    short_term_rows.append({
        "student_id": student_id,
        "session_id": session_id,
        "current_concept": current_concept,
        "recent_interaction_count": interaction_count,
        "recent_accuracy": round(recent_accuracy, 2),
        "average_attempts": round(average_attempts, 2),
        "recent_hint_usage": recent_hint_usage,
        "session_status": session_status
    })

short_term_memory_df = pd.DataFrame(short_term_rows)

print("Short-term memory table created.")
print("Shape:", short_term_memory_df.shape)

short_term_memory_df.head()

Short-term memory table created.
Shape: (4151, 8)


,student_id,session_id,current_concept,recent_interaction_count,recent_accuracy,average_attempts,recent_hint_usage,session_status
0,74864,261340,Addition and Subtraction Positive Decimals,1,1.00,1.00,0,good_progress
1,78155,262440,Multiplication and Division Integers,7,0.71,1.57,1,moderate_progress
2,78271,262440,Multiplication and Division Integers,1,0.00,1.00,0,needs_support
3,78733,262440,Multiplication and Division Integers,1,0.00,0.00,0,needs_support
4,70657,220671,Range,4,0.00,1.00,0,needs_support


In [19]:
#Part B: Generate Long-Term Memory Table

#Long-term memory should summarize each student’s behavior across all sessions.



In [20]:
# Create Concept Summary per Student

student_concept_summary = df.groupby(["student_id", "concept_name"]).agg(
    total_interactions=("correct", "count"),
    correct_count=("correct", "sum"),
    avg_hints=("hint_count", "mean"),
    total_hints=("hint_count", "sum"),
    avg_attempts=("attempt_count", "mean"),
    avg_response_time_ms=("response_time_ms", "mean")
).reset_index()

student_concept_summary["wrong_count"] = (
    student_concept_summary["total_interactions"] - student_concept_summary["correct_count"]
)

student_concept_summary["accuracy"] = (
    student_concept_summary["correct_count"] / student_concept_summary["total_interactions"]
)

student_concept_summary.head()

,student_id,concept_name,total_interactions,correct_count,avg_hints,total_hints,avg_attempts,avg_response_time_ms,wrong_count,accuracy
0,14,Circle Graph,12,3,1.083333,13,0.75,32656.750000,9,0.250000
1,14,Equivalent Fractions,6,2,0.500000,3,0.50,47747.333333,4,0.333333
2,14,Finding Percents,2,0,1.500000,3,0.50,24554.500000,2,0.000000
3,14,Median,4,0,2.500000,10,1.00,6697.250000,4,0.000000
4,14,Percent Of,6,1,1.666667,10,1.00,17566.166667,5,0.166667


In [21]:
#Generate Weak and Strong Areas

weak_area_df = student_concept_summary[
    (student_concept_summary["accuracy"] < 0.50) &
    (student_concept_summary["total_interactions"] >= 3)
]

strong_area_df = student_concept_summary[
    (student_concept_summary["accuracy"] >= 0.75) &
    (student_concept_summary["total_interactions"] >= 3)
]

weak_areas_by_student = weak_area_df.groupby("student_id")["concept_name"].apply(list).reset_index()
weak_areas_by_student = weak_areas_by_student.rename(columns={"concept_name": "weak_areas"})

strong_areas_by_student = strong_area_df.groupby("student_id")["concept_name"].apply(list).reset_index()
strong_areas_by_student = strong_areas_by_student.rename(columns={"concept_name": "strong_areas"})

weak_areas_by_student.head()

,student_id,weak_areas
0,14,"[Circle Graph, Equivalent Fractions, Median, P..."
1,21825,[Median]
2,53167,"[Area Circle, Estimation, Order of Operations ..."
3,58161,[Equation Solving More Than Two Steps]
4,64634,[Multiplication Fractions]


In [22]:
#Create Long-Term Memory for All Students

long_term_memory_df = df.groupby("student_id").agg(
    total_interactions=("correct", "count"),
    total_sessions=("session_id", "nunique"),
    total_concepts=("concept_name", "nunique"),
    overall_accuracy=("correct", "mean"),
    average_attempts=("attempt_count", "mean"),
    average_hint_count=("hint_count", "mean"),
    total_hint_count=("hint_count", "sum"),
    average_response_time_ms=("response_time_ms", "mean")
).reset_index()

long_term_memory_df = long_term_memory_df.merge(
    weak_areas_by_student,
    on="student_id",
    how="left"
)

long_term_memory_df = long_term_memory_df.merge(
    strong_areas_by_student,
    on="student_id",
    how="left"
)

long_term_memory_df["weak_areas"] = long_term_memory_df["weak_areas"].apply(
    lambda x: x if isinstance(x, list) else []
)

long_term_memory_df["strong_areas"] = long_term_memory_df["strong_areas"].apply(
    lambda x: x if isinstance(x, list) else []
)

def assign_support_style(row):
    if row["average_hint_count"] >= 2:
        return "guided_support"
    elif row["average_attempts"] >= 2:
        return "step_by_step_support"
    elif row["overall_accuracy"] >= 0.75:
        return "independent_practice"
    else:
        return "basic_explanation_support"

long_term_memory_df["preferred_support_style"] = long_term_memory_df.apply(
    assign_support_style,
    axis=1
)

long_term_memory_df["overall_accuracy"] = long_term_memory_df["overall_accuracy"].round(2)
long_term_memory_df["average_attempts"] = long_term_memory_df["average_attempts"].round(2)
long_term_memory_df["average_hint_count"] = long_term_memory_df["average_hint_count"].round(2)
long_term_memory_df["average_response_time_ms"] = long_term_memory_df["average_response_time_ms"].round(2)

print("Long-term memory table created.")
print("Shape:", long_term_memory_df.shape)

long_term_memory_df.head()

Long-term memory table created.
Shape: (4151, 12)


,student_id,total_interactions,total_sessions,total_concepts,overall_accuracy,average_attempts,average_hint_count,total_hint_count,average_response_time_ms,weak_areas,strong_areas,preferred_support_style
0,14,35,3,7,0.26,0.74,1.11,39,29605.34,"[Circle Graph, Equivalent Fractions, Median, P...",[Range],basic_explanation_support
1,21825,24,4,4,0.75,0.92,0.00,0,59186.96,[Median],"[Mean, Multiplication and Division Integers]",independent_practice
2,51950,6,2,2,0.83,1.83,0.50,3,61343.00,[],[Equation Solving Two or Fewer Steps],independent_practice
3,52613,7,4,5,0.57,0.57,1.14,8,14414.29,[],[Distributive Property],basic_explanation_support
4,53167,312,102,53,0.70,1.58,0.16,50,29769.09,"[Area Circle, Estimation, Order of Operations ...","[Absolute Value, Addition Whole Numbers, Addit...",basic_explanation_support


In [23]:
#Part C: Generate Concept-Based Memory Table

#This is concept-level interaction history.

In [24]:
#Create Concept-Based Memory Table

concept_based_memory_df = student_concept_summary.copy()

def identify_observed_behavior(row):
    if row["accuracy"] < 0.50 and row["total_interactions"] >= 3:
        return "repeated_difficulty"
    elif row["accuracy"] >= 0.75 and row["total_interactions"] >= 3:
        return "consistent_strength"
    elif row["avg_hints"] >= 2:
        return "hint_dependent"
    elif row["avg_attempts"] >= 2:
        return "needs_more_attempts"
    else:
        return "normal_progress"

concept_based_memory_df["observed_behavior"] = concept_based_memory_df.apply(
    identify_observed_behavior,
    axis=1
)

def generate_memory_note(row):
    if row["observed_behavior"] == "repeated_difficulty":
        return "Student has repeated difficulty in this concept. Provide simpler explanation and more examples."
    elif row["observed_behavior"] == "hint_dependent":
        return "Student often uses hints in this concept. Provide guided hints before final answers."
    elif row["observed_behavior"] == "needs_more_attempts":
        return "Student needs multiple attempts in this concept. Break tasks into smaller steps."
    elif row["observed_behavior"] == "consistent_strength":
        return "Student performs strongly in this concept. Provide less guidance or advanced practice."
    else:
        return "Student shows normal progress. Continue standard support."

concept_based_memory_df["memory_note"] = concept_based_memory_df.apply(
    generate_memory_note,
    axis=1
)

concept_based_memory_df["accuracy"] = concept_based_memory_df["accuracy"].round(2)
concept_based_memory_df["avg_hints"] = concept_based_memory_df["avg_hints"].round(2)
concept_based_memory_df["avg_attempts"] = concept_based_memory_df["avg_attempts"].round(2)
concept_based_memory_df["avg_response_time_ms"] = concept_based_memory_df["avg_response_time_ms"].round(2)

print("Concept-based memory table created.")
print("Shape:", concept_based_memory_df.shape)

concept_based_memory_df.head()

Concept-based memory table created.
Shape: (39309, 12)


,student_id,concept_name,total_interactions,correct_count,avg_hints,total_hints,avg_attempts,avg_response_time_ms,wrong_count,accuracy,observed_behavior,memory_note
0,14,Circle Graph,12,3,1.08,13,0.75,32656.75,9,0.25,repeated_difficulty,Student has repeated difficulty in this concep...
1,14,Equivalent Fractions,6,2,0.50,3,0.50,47747.33,4,0.33,repeated_difficulty,Student has repeated difficulty in this concep...
2,14,Finding Percents,2,0,1.50,3,0.50,24554.50,2,0.00,normal_progress,Student shows normal progress. Continue standa...
3,14,Median,4,0,2.50,10,1.00,6697.25,4,0.00,repeated_difficulty,Student has repeated difficulty in this concep...
4,14,Percent Of,6,1,1.67,10,1.00,17566.17,5,0.17,repeated_difficulty,Student has repeated difficulty in this concep...


In [25]:
#Part D: Save Memory Tables

#Save All Memory Tables

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../outputs/tables", exist_ok=True)

short_term_memory_df.to_csv(
    "../data/processed/short_term_memory.csv",
    index=False
)

long_term_memory_df.to_csv(
    "../data/processed/long_term_memory.csv",
    index=False
)

concept_based_memory_df.to_csv(
    "../data/processed/concept_based_memory.csv",
    index=False
)

# Also save copies in outputs for evidence
short_term_memory_df.head(50).to_csv(
    "../outputs/tables/sample_short_term_memory.csv",
    index=False
)

long_term_memory_df.head(50).to_csv(
    "../outputs/tables/sample_long_term_memory.csv",
    index=False
)

concept_based_memory_df.head(100).to_csv(
    "../outputs/tables/sample_concept_based_memory.csv",
    index=False
)

print("All memory tables saved successfully.")

All memory tables saved successfully.


In [26]:
#Part E: Generate Sample Memory Context JSON

#This is the output your Memory Component provides to other agents.

#Generate Memory Context Function

def get_memory_context(student_id, target_concept=None):
    short_term = short_term_memory_df[
        short_term_memory_df["student_id"] == student_id
    ]
    
    long_term = long_term_memory_df[
        long_term_memory_df["student_id"] == student_id
    ]
    
    concept_records = concept_based_memory_df[
        concept_based_memory_df["student_id"] == student_id
    ]
    
    if short_term.empty:
        return {"error": "Student not found in short-term memory"}
    
    if long_term.empty:
        return {"error": "Student not found in long-term memory"}
    
    short_term_dict = short_term.iloc[0].to_dict()
    long_term_dict = long_term.iloc[0].to_dict()
    
    if target_concept is None:
        if len(long_term_dict["weak_areas"]) > 0:
            target_concept = long_term_dict["weak_areas"][0]
        else:
            target_concept = concept_records.iloc[0]["concept_name"]
    
    target_concept_record = concept_records[
        concept_records["concept_name"] == target_concept
    ]
    
    if target_concept_record.empty:
        concept_dict = {
            "concept_name": target_concept,
            "message": "No concept-based memory found for this student and concept."
        }
    else:
        concept_dict = target_concept_record.iloc[0].to_dict()
    
    memory_context = {
        "student_id": int(student_id),
        "target_concept": target_concept,
        "short_term_memory": short_term_dict,
        "long_term_memory": long_term_dict,
        "concept_based_memory": concept_dict,
        "personalization_context_for_agents": {
            "student_support_style": long_term_dict["preferred_support_style"],
            "weak_areas": long_term_dict["weak_areas"],
            "strong_areas": long_term_dict["strong_areas"],
            "recommended_agent_usage": "Use this memory context to personalize explanation, planning, evaluation, and meta-level analysis."
        }
    }
    
    return memory_context

In [27]:
#Test Memory Context

sample_student_id = short_term_memory_df["student_id"].iloc[0]

sample_memory_context = get_memory_context(sample_student_id)

print(json.dumps(sample_memory_context, indent=4, default=str))

{
    "student_id": 74864,
    "target_concept": "Addition and Subtraction Positive Decimals",
    "short_term_memory": {
        "student_id": 74864,
        "session_id": 261340,
        "current_concept": "Addition and Subtraction Positive Decimals",
        "recent_interaction_count": 1,
        "recent_accuracy": 1.0,
        "average_attempts": 1.0,
        "recent_hint_usage": 0,
        "session_status": "good_progress"
    },
    "long_term_memory": {
        "student_id": 74864,
        "total_interactions": 1,
        "total_sessions": 1,
        "total_concepts": 1,
        "overall_accuracy": 1.0,
        "average_attempts": 1.0,
        "average_hint_count": 0.0,
        "total_hint_count": 0,
        "average_response_time_ms": 18720.0,
        "weak_areas": [],
        "strong_areas": [],
        "preferred_support_style": "independent_practice"
    },
    "concept_based_memory": {
        "student_id": 74864,
        "concept_name": "Addition and Subtraction Positive D

In [28]:
#Save Sample Memory Context JSON

os.makedirs("../outputs/api_results", exist_ok=True)

with open("../outputs/api_results/sample_memory_context.json", "w") as f:
    json.dump(sample_memory_context, f, indent=4, default=str)

print("Sample memory context JSON saved successfully.")

Sample memory context JSON saved successfully.
